# Convert figured-bass `startid` anchors to `tstamp`

This notebook demonstrates the workflow implemented in `scripts/convert_harm_startid_to_tstamp.py` using `bsb00023199_00023_facs_zones.mei`.

The converter changes parent `<harm startid="#…">` anchors to `<harm tstamp="…" staff="…">`. The nested `<fb>` and `<f>` content is preserved. The example first performs a dry run, then applies the conversion only to a temporary copy in `/tmp`, validates the XML, and leaves the source MEI untouched.

In [1]:
from pathlib import Path
import shutil
import sys
import tempfile
import xml.etree.ElementTree as ET

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
SCRIPTS_DIR = ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from convert_harm_startid_to_tstamp import (
    build_rows,
    convert_file,
    convert_text,
    normalize_figured_bass_accidentals,
)

SOURCE = ROOT / "11_buxtehude_dietrich_buxtehudes_instrumentalwerke_bsb00023199" / "bsb00023199_00023_facs_zones.mei"
assert SOURCE.exists(), SOURCE
SOURCE

PosixPath('/home/eggi/Nextcloud/code/public_repos/camat_corpus_editions/DdT_1_vol_11/11_buxtehude_dietrich_buxtehudes_instrumentalwerke_bsb00023199/bsb00023199_00023_facs_zones.mei')

## 1. Inspect the current attachment styles

Only `<harm>` elements containing `<fb>` are counted. Some are already encoded with `staff+tstamp`; the rest use `startid`.

In [2]:
def local_name(tag):
    return tag.rsplit("}", 1)[-1]

def figured_bass_harms(path):
    root = ET.parse(path).getroot()
    return [
        element
        for element in root.iter()
        if local_name(element.tag) == "harm"
        and any(local_name(child.tag) == "fb" for child in element)
    ]

source_harms = figured_bass_harms(SOURCE)
source_summary = {
    "figured-bass harm elements": len(source_harms),
    "with startid": sum("startid" in harm.attrib for harm in source_harms),
    "with tstamp": sum("tstamp" in harm.attrib for harm in source_harms),
    "with staff": sum("staff" in harm.attrib for harm in source_harms),
}
pd.Series(source_summary, name="before conversion")

figured-bass harm elements    39
with startid                   0
with tstamp                   39
with staff                    39
Name: before conversion, dtype: int64

## 2. Dry run and review the calculated timestamps

`apply=False` calculates every proposed change but does not write the MEI file. Timestamps are one-based MEI beat positions. The target staff is derived from the referenced note, rest, space, or chord unless `<harm>` already has an explicit `staff`.

`normalize_accidentals=True` also checks figured-bass `<f>` text and proposes Unicode `♭` and `♯` in place of ASCII `b` and `#`.

In [3]:
dry_run = convert_file(SOURCE, apply=False, normalize_accidentals=True)
print(
    f"converted={dry_run.converted}, accidentals={dry_run.normalized_accidentals}, "
    f"skipped={dry_run.skipped}, "
    f"unresolved={dry_run.unresolved}, source_written={dry_run.applied}"
)

review = pd.DataFrame([row.__dict__ for row in dry_run.rows])
display(review[[
    "line", "status", "measure_n", "startid", "target_kind",
    "target_staff", "tstamp", "figure_text", "normalized_figure", "message"
]])

converted=0, accidentals=6, skipped=0, unresolved=0, source_written=False


,line,status,measure_n,startid,target_kind,target_staff,tstamp,figure_text,normalized_figure,message
0,470,normalize_accidental,,,,,,4#,4♯,ASCII figured-bass accidental should use a mus...
1,739,normalize_accidental,,,,,,4#,4♯,ASCII figured-bass accidental should use a mus...
2,854,normalize_accidental,,,,,,6#,6♯,ASCII figured-bass accidental should use a mus...
3,1007,normalize_accidental,,,,,,#,♯,ASCII figured-bass accidental should use a mus...
4,1233,normalize_accidental,,,,,,#,♯,ASCII figured-bass accidental should use a mus...
5,1481,normalize_accidental,,,,,,5b,5♭,ASCII figured-bass accidental should use a mus...


## 3. Preview the rewritten `<harm>` opening tags

The converter preserves the rest of the source text. This compact preview shows only lines whose `<harm>` opening tag changes.

In [4]:
source_text = SOURCE.read_text(encoding="utf-8")
rows, rows_by_index = build_rows(SOURCE, source_text)
converted_text, converted_count = convert_text(source_text, rows_by_index)
converted_text, accidental_rows = normalize_figured_bass_accidentals(SOURCE, converted_text)

changed_lines = []
for line_number, (before, after) in enumerate(
    zip(source_text.splitlines(), converted_text.splitlines()), start=1
):
    if before != after:
        changed_lines.append({
            "line": line_number,
            "before": before.strip(),
            "after": after.strip(),
        })

assert len(changed_lines) == converted_count + len(accidental_rows)
display(pd.DataFrame(changed_lines).head(12))
print(
    f"Showing 12 of {converted_count} anchor changes and "
    f"{len(accidental_rows)} accidental changes."
)

,line,before,after
0,470,<f> 4#</f>,<f> 4♯</f>
1,739,<f>4#</f>,<f>4♯</f>
2,854,<f>6#</f>,<f>6♯</f>
3,1007,<f>#</f>,<f>♯</f>
4,1233,<f>#</f>,<f>♯</f>
5,1481,<f>5b</f>,<f>5♭</f>


Showing 12 of 0 anchor changes and 6 accidental changes.


## 4. Apply the conversion to a temporary copy

This is the safe rehearsal of `--apply`. The original file remains unchanged.

In [5]:
WORK_DIR = Path(tempfile.mkdtemp(prefix="fb-startid-to-tstamp-", dir="/tmp"))
OUTPUT = WORK_DIR / SOURCE.name
shutil.copy2(SOURCE, OUTPUT)

applied = convert_file(OUTPUT, apply=True, normalize_accidentals=True)
print(
    f"converted={applied.converted}, accidentals={applied.normalized_accidentals}, "
    f"skipped={applied.skipped}, "
    f"unresolved={applied.unresolved}, written={applied.applied}"
)
print(f"Temporary converted file: {OUTPUT}")

converted=0, accidentals=6, skipped=0, unresolved=0, written=True
Temporary converted file: /tmp/fb-startid-to-tstamp-w7_lcisv/bsb00023199_00023_facs_zones.mei


## 5. Validate the result

The checks below verify well-formed XML, confirm that every figured-bass `<harm>` has `tstamp` and `staff`, confirm that no `startid` remains on those elements, and confirm that the source file was not modified.

In [6]:
converted_harms = figured_bass_harms(OUTPUT)
validation = {
    "well-formed XML": True,
    "figured-bass harm elements": len(converted_harms),
    "remaining startid": sum("startid" in harm.attrib for harm in converted_harms),
    "with tstamp": sum("tstamp" in harm.attrib for harm in converted_harms),
    "with staff": sum("staff" in harm.attrib for harm in converted_harms),
    "ASCII accidental remnants": sum(
        "#" in (figure.text or "") or "b" in (figure.text or "")
        for harm in converted_harms
        for figure in harm.iter()
        if local_name(figure.tag) == "f"
    ),
    "source unchanged": SOURCE.read_text(encoding="utf-8") == source_text,
}

assert validation["remaining startid"] == 0
assert validation["with tstamp"] == len(converted_harms)
assert validation["with staff"] == len(converted_harms)
assert validation["ASCII accidental remnants"] == 0
assert validation["source unchanged"]
pd.Series(validation, name="after conversion")

well-formed XML               True
figured-bass harm elements      39
remaining startid                0
with tstamp                     39
with staff                      39
ASCII accidental remnants        0
source unchanged              True
Name: after conversion, dtype: object

## 6. Command-line workflow

Run the converter without `--apply` first:

```bash
python scripts/convert_harm_startid_to_tstamp.py \
  11_buxtehude_dietrich_buxtehudes_instrumentalwerke_bsb00023199/bsb00023199_00023_facs_zones.mei \
  --normalize-accidentals \
  --report /tmp/fb_tstamp_report.csv
```

After reviewing the report, add `--apply` to rewrite the selected file:

```bash
python scripts/convert_harm_startid_to_tstamp.py \
  11_buxtehude_dietrich_buxtehudes_instrumentalwerke_bsb00023199/bsb00023199_00023_facs_zones.mei \
  --apply \
  --normalize-accidentals \
  --report /tmp/fb_tstamp_applied_report.csv
```

Multiple MEI paths may be supplied in one command. Cross-measure or unresolved references are reported and left unchanged for manual inspection. Without `--apply`, accidental changes are only reported; with `--apply`, ASCII `b/#` inside figured-bass `<f>` text are rewritten as `♭/♯`.